# 🚀 Viettel AI Race 2026 — v6.0 Validation Notebook
**Mục đích**: Xác nhận `--quantization=fp8` + `--kv-cache-dtype=fp8_e4m3` hoạt động với LFM2.5-1.2B trước khi dùng lượt submit cuối.

**Checklist cần pass trước khi submit**:
- [ ] Server khởi động không lỗi (compatibility check)
- [ ] Accuracy sanity check (không bị 0 điểm do accuracy drop)
- [ ] Server trả về streaming response bình thường

## Bước 1: Kiểm tra CUDA version & Cài đúng vLLM

> **Lý do cài version cụ thể**: `pip install vllm` mặc định cài version mới nhất (0.9+) yêu cầu `libcudart.so.13` (CUDA 13), trong khi Colab T4 chỉ có CUDA 12.x → gây lỗi `ImportError`. Cần cài `vllm==0.6.4.post1` tương thích CUDA 12.x.

In [ ]:
# Kiểm tra CUDA version
!nvcc --version
!nvidia-smi

# Cài đúng phiên bản vLLM tương thích CUDA 12.x trên Colab T4
# KHÔNG dùng 'pip install vllm' (cài mới nhất yêu cầu CUDA 13 -> lỗi)
!pip install -q 'vllm==0.6.4.post1' aiohttp openai numpy huggingface_hub

import torch
print(f"\nPyTorch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("❌ Không có GPU! Vào Runtime → Change runtime type → chọn T4 GPU")

## Bước 2: Tải Model

In [ ]:
import os
from huggingface_hub import snapshot_download

model_dir = "./model"
if not os.path.exists(model_dir):
    print("Downloading LiquidAI/LFM2.5-1.2B-Instruct...")
    snapshot_download(repo_id="LiquidAI/LFM2.5-1.2B-Instruct", local_dir=model_dir)
    print("Done!")
else:
    print("Model already at ./model")

## Bước 3: Khởi chạy vLLM Server — Cấu hình v6.0

**Đây là config sẽ submit**: `--quantization=fp8` + `--kv-cache-dtype=fp8_e4m3`

> ⚠️ Trên T4, FP8 chạy bằng phần mềm (không có hardware tensor core FP8).  
> Mục đích ở đây là **kiểm tra compatibility**, không phải đo hiệu năng thực tế.  
> Nếu server khởi động OK ở đây → trên H200 với Docker cũng sẽ OK.

In [ ]:
import subprocess, time, requests, torch

gpu_name = torch.cuda.get_device_name(0)
is_t4 = "T4" in gpu_name

# === CẤU HÌNH V6.0 (giống docker-compose.yml sẽ submit) ===
vllm_cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model=./model",
    "--served-model-name=LFM2.5-1.2B-Instruct",
    "--host=0.0.0.0",
    "--port=8000",
    "--tensor-parallel-size=1",
    "--max-model-len=4096" if is_t4 else "--max-model-len=8192",
    "--gpu-memory-utilization=0.85" if is_t4 else "--gpu-memory-utilization=0.97",
    "--enable-prefix-caching",
    # === HAI DÒNG QUAN TRỌNG CẦN XÁC NHẬN ===
    "--quantization=fp8",
    "--kv-cache-dtype=fp8_e4m3",
]

print(f"GPU: {gpu_name}")
print("Config v6.0: FP8 weight + FP8 KV cache (e4m3)")
print("Command:", ' '.join(vllm_cmd))

!pkill -f "vllm.entrypoints.openai.api_server"
time.sleep(2)

log_file = open("vllm_v6.log", "w")
server_process = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=log_file)
print("\nServer starting... (chờ tối đa 120s)")

ready = False
for i in range(60):
    try:
        resp = requests.get("http://localhost:8000/health", timeout=2)
        if resp.status_code == 200:
            print(f"\n✅ Server sẵn sàng sau {(i+1)*2}s!")
            ready = True
            break
    except:
        pass
    if (i+1) % 5 == 0:
        print(f"  Đang chờ... {(i+1)*2}s")
    time.sleep(2)

if not ready:
    print("\n❌ SERVER KHÔNG KHỞI ĐỘNG ĐƯỢC!")
    print("=== LOG LỖI (2000 ký tự cuối) ===")
    log_file.close()
    with open("vllm_v6.log", "r") as f:
        content = f.read()
        print(content[-2000:])
    print("\n⛔ KẾT LUẬN: Xem log bên trên để chẩn đoán.")
else:
    print("✅ PASS: Server khởi động OK — FP8 weight + FP8 KV e4m3 TƯƠNG THÍCH với LFM2.5!")

## Bước 4: Accuracy Sanity Check

In [ ]:
import requests

GPQA_STYLE_QUESTIONS = [
    {"q": "What is the speed of light in vacuum?\nA) 3x10^8 m/s\nB) 3x10^6 m/s\nC) 3x10^10 m/s\nD) 3x10^4 m/s", "a": "A"},
    {"q": "Which organelle is responsible for ATP synthesis in eukaryotic cells?\nA) Ribosome\nB) Lysosome\nC) Mitochondria\nD) Golgi apparatus", "a": "C"},
    {"q": "What is the derivative of sin(x)?\nA) -cos(x)\nB) cos(x)\nC) tan(x)\nD) -sin(x)", "a": "B"},
    {"q": "In quantum mechanics, what does the Heisenberg Uncertainty Principle state?\nA) Energy is conserved\nB) Position and momentum cannot both be precisely measured\nC) Light travels in straight lines\nD) Entropy always increases", "a": "B"},
    {"q": "Which element has atomic number 79?\nA) Silver\nB) Platinum\nC) Gold\nD) Mercury", "a": "C"},
    {"q": "DNA replication is:\nA) Conservative\nB) Dispersive\nC) Semi-conservative\nD) Random", "a": "C"},
    {"q": "The Pauli exclusion principle states that:\nA) No two fermions can occupy the same quantum state\nB) Energy is quantized\nC) Light has wave-particle duality\nD) Mass-energy equivalence", "a": "A"},
    {"q": "What is the powerhouse of the cell?\nA) Nucleus\nB) Mitochondria\nC) Ribosome\nD) Cell membrane", "a": "B"},
]

def ask_model(question):
    resp = requests.post("http://localhost:8000/v1/chat/completions", json={
        "model": "LFM2.5-1.2B-Instruct",
        "messages": [{"role": "user", "content": f"Answer with ONLY the letter (A, B, C, or D).\n\n{question}"}],
        "max_tokens": 5,
        "temperature": 0.0
    }, timeout=30)
    return resp.json()["choices"][0]["message"]["content"].strip()

correct = 0
print("=== ACCURACY RESULTS ===")
print(f"{'Q':>3} | Expected | Got    | Status")
print("-" * 40)
for i, item in enumerate(GPQA_STYLE_QUESTIONS):
    try:
        answer = ask_model(item["q"])
        ok = item["a"] in answer
        correct += ok
        print(f"{i+1:>3} | {item['a']:^8} | {answer[:6]:^6} | {'✅' if ok else '❌'}")
    except Exception as e:
        print(f"{i+1:>3} | {item['a']:^8} | {'ERROR':^6} | 💥 {e}")

accuracy = correct / len(GPQA_STYLE_QUESTIONS)
print("-" * 40)
print(f"Accuracy: {correct}/{len(GPQA_STYLE_QUESTIONS)} = {accuracy:.1%}")
print()
if accuracy >= 0.625:
    print("✅ PASS: Accuracy đủ tốt — FP8 KV cache AN TOÀN")
else:
    print("⚠️  WARN: Accuracy thấp — kiểm tra lại trước khi submit")

## Bước 5: Streaming Response Check

In [ ]:
import requests, json, time

print("=== STREAMING TEST ===")
t_start = time.perf_counter()
first_token_time = None
token_times = []

resp = requests.post("http://localhost:8000/v1/chat/completions", json={
    "model": "LFM2.5-1.2B-Instruct",
    "messages": [
        {"role": "system", "content": "You are a helpful AI assistant. " * 50},
        {"role": "user", "content": "Explain briefly what is LLM inference optimization."}
    ],
    "max_tokens": 100,
    "stream": True,
    "temperature": 0.1
}, stream=True, timeout=60)

output = ""
for line in resp.iter_lines():
    if line:
        line = line.decode('utf-8')
        if line.startswith("data: ") and line[6:] != "[DONE]":
            try:
                data = json.loads(line[6:])
                content = data.get("choices", [{}])[0].get("delta", {}).get("content", "")
                if content:
                    t_now = time.perf_counter()
                    if first_token_time is None:
                        first_token_time = t_now
                    token_times.append(t_now)
                    output += content
            except:
                pass

token_count = len(token_times)
ttft = (first_token_time - t_start) * 1000 if first_token_time else 0
tbt = (token_times[-1] - first_token_time) / max(1, token_count - 1) * 1000 if token_count > 1 else 0

print(f"TTFT: {ttft:.1f}ms")
print(f"TBT: {tbt:.1f}ms  (note: T4 is slower than H200, not representative)")
print(f"Tokens generated: {token_count}")
print(f"Output: {output[:200]}")
print()
if token_count > 0:
    print("✅ PASS: Streaming OK")
else:
    print("❌ FAIL: Không nhận được token nào!")

## Bước 6: Tổng kết GO / NO-GO

In [ ]:
print("=" * 50)
print("CHECKLIST SUBMIT v6.0 (1 lượt cuối)")
print("=" * 50)

server_ok    = input("Bước 3 — Server khởi động OK? (y/n): ").strip().lower() == 'y'
accuracy_ok  = input("Bước 4 — Accuracy >= 5/8 (62.5%)? (y/n): ").strip().lower() == 'y'
streaming_ok = input("Bước 5 — Streaming OK? (y/n): ").strip().lower() == 'y'

print()
print("=" * 50)
if server_ok and accuracy_ok and streaming_ok:
    print("🟢 GO: AN TOÀN ĐỂ SUBMIT v6.0")
    print("   → Nộp docker-compose.yml ngay!")
    print("   → Dự kiến: TBT 4ms → 2ms → score ~77+")
elif not server_ok:
    print("🔴 NO-GO: Server không chạy → KHÔNG SUBMIT v6.0")
    print("   → Giữ nguyên v5.0 (61.15 điểm)")
elif not accuracy_ok:
    print("🔴 NO-GO: Accuracy thấp → Nguy cơ f(Δ)=0")
    print("   → Giữ nguyên v5.0 (61.15 điểm)")
else:
    print("🟡 CÂN NHẮC: Server OK + Accuracy OK nhưng streaming có vấn đề nhỏ")
    print("   → Vẫn có thể submit, BTC test streaming riêng")